In [152]:
# Import dependencies

import os
import wandb
import wandb_workspaces.workspaces as ws
import wandb_workspaces.reports.v2 as wr # We use the Reports API for adding panels

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [153]:
WANDB_API_KEY = "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"
ENTITY = "kirill456z"
PROJECT = "physics4llm"

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY

In [154]:
#import wandb_workspaces.reports.v2 as wr

#report = wr.Report(
     #entity=ENTITY,
     #project=PROJECT,
     #title="midterm report",
     #description="A descriptive description.",
#)

#report.save()

In [155]:
report = wr.Report.from_url("https://wandb.ai/kirill456z/physics4llm/reports/midterm-report--VmlldzoxNjM3NTU5NA==")

report.width = 'fluid'

In [156]:
from plots import loss_lineplot, canon_weight_fixed_layer, canon_rms_ratio, residual_rms, depo_eval_plot
from plots import grad_contrib, outlier_features_kurtosis, cos_sim, realign_grid, to_matplotlib
from runsets import RunsetsFactory

runsets_factory = RunsetsFactory(ENTITY, PROJECT)
FAST_PLOT_KWARGS = dict(api_timeout=90, max_points_per_series=150, use_scan_fallback=False)

report.blocks = []

## F1 — Synthetic Tasks Are Too Noisy to Compare Convergence Speed

In [157]:
# F1 — Text loss stability across seeds
text_run_names = [
    "text_llama_bs_128_seq_len_2048_0.1.136",
    "canon_text_0.1.130",
    "text_ks_4_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.147",
    "text_ks_4_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.140",
]
text_runs = [runsets_factory.filter_by_run_names(text_run_names)]

text_loss_plots = [loss_lineplot(title="Text Loss — seed variance")]

# F1 — Depo 8L512D stability across seeds
depo_run_names = [
    "depo_ks_4_default_init_with_residual_trainable_0.1.122",
    "depo_ks_4_default_init_with_residual_trainable_0.1.121",
    "depo_ks_4_default_init_with_residual_trainable_0.1.120",
    "depo_llama_8_hops_100_nodes_bs_256_seq_len_768_0.1.166",
    "llama_0.1.126",
]
depo_runs = [runsets_factory.filter_by_run_names(depo_run_names)]

depo_loss_plots = [loss_lineplot(title="Depo Loss 8L512D — seed variance")]
depo_biger_model_loss_plots = [loss_lineplot(title = "Depo Loss 12L768D")]
depo_eval_plots = [depo_eval_plot(title="Depo 8L512D - eval")]

# F1 - depo bigger model
depo_biger_model = [
    "depo_ks_4_default_init_with_residual_trainable_8_hops_100_nodes_bs_128_seq_len_1024_0.1.156",
    "depo_ks_4_default_init_with_residual_trainable_8_hops_100_nodes_bs_128_seq_len_1024_0.1.155"
]

depo_biger_model_runs = [runsets_factory.filter_by_run_names(depo_biger_model)]


# F1 — Brevo 110-node divergence as additional variance evidence
brevo_run_names = [
    "brevo_llama_110_nodes_single_bs_256_seq_len_1024_0.1.219",
    "brevo_llama_110_nodes_single_bs_256_seq_len_1024_0.1.218",
    "brevo_llama_110_nodes_single_bs_256_seq_len_1024_0.1.217",
]
brevo_runs = [runsets_factory.filter_by_run_names(brevo_run_names)]

brevo_loss_plots = [loss_lineplot(title="Brevo Loss 110 nodes — divergent runs", y_max=3)]

report.blocks += [
    wr.H1("Findings"),
    wr.H2("F1 — Training Is Noisy: Seed Variance Dominates Convergence Speed"),
    wr.P(
        "Loss convergence speed varies significantly with model initialization seeds on syntetic tasks rendering them unsuitable to study training dynamics"
    ),
    wr.H3("Depo 8L512D — grokking jumps at different steps per seed"),
    wr.P(
        "Leveled nature of the tasks results in grokking-style loss drops, which occur at "
        "different training steps across seeds. In some cases llama seed may converge "
        "faster than canon — not because Canon is worse, but because "
        "the task's loss landscape is sensitive to random initialization. "
        "A single-seed comparison could give a misleading ranking."
    ),
    wr.PanelGrid(runsets=depo_runs, panels=depo_loss_plots, hide_run_sets=True),
    wr.PanelGrid(runsets=depo_runs, panels=depo_eval_plots, hide_run_sets=True),
    wr.P("Similar divergence with a bigger model:"),
    wr.PanelGrid(runsets=depo_biger_model_runs, panels = depo_biger_model_loss_plots, hide_run_sets = True),
    wr.H3("And same for Brevo task:"),
    wr.PanelGrid(runsets=brevo_runs, panels=brevo_loss_plots, hide_run_sets=True),
    wr.H3("Text loss"),
    wr.P(
        "For text modeling different seeds produce identical curves. "
        "Canon performs better than baseline"
    ),
    wr.PanelGrid(runsets=text_runs, panels=text_loss_plots, hide_run_sets=True),
    wr.P(
        "Conclusion: synthetic tasks are useful for final-accuracy comparisons across "
        "multiple seeds, but a single-seed convergence curve is unreliable. "
        "Multiple runs need to be aggregated draw conclusions"
    ),
]

## F2 — Canon Initialization Matters

In [158]:
init_ablation_run_names = [
    "text_llama_bs_128_seq_len_2048_0.1.136",                                        # Baseline (no Canon)
    "text_ks_4_zeros_init_with_residual_trainable_bs_128_seq_len_2048_0.1.162",      # Zero init
    "text_ks_4_const_var_init_with_residual_trainable_bs_128_seq_len_2048_0.1.163",  # const_var (1/sqrt(k))
    "text_ks_4_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.140",  # const_var_sqrt (1/k)
    "canon_text_0.1.129",                                                             # Default random init
]
init_ablation_runs = [runsets_factory.filter_by_run_names(init_ablation_run_names)]

init_ablation_plots = [loss_lineplot(title="Canon Initialization Ablation")]

report.blocks += [
    wr.H2("F2 — Canon Initialization Matters"),
    wr.P(
        "The initial values of the Canon conv1d weights have a measurable effect on "
        "final text loss. Five configurations were compared (k — conv1d kernel size):"
    ),
    wr.LatexBlock(r"\text{zeros}: \quad w_0 = 0"),
    wr.LatexBlock(r"\text{const\_var}: \quad w_0 = \frac{1}{\sqrt{k}}"),
    wr.LatexBlock(r"\text{const\_var\_sqr}: \quad w_0 = \frac{1}{k}"),
    wr.LatexBlock(r"\text{default}: \quad w_0 \sim \mathcal{U}\!\left[-\frac{1}{\sqrt{k}},\; \frac{1}{\sqrt{k}}\right]"),
    wr.LatexBlock(r"\text{baseline}: \quad \text{no Canon}"),
    wr.P(
        "loss ranking: "
        "const_var_sqr < const_var < default < zeros < baseline — "
        "confirming Canon always helps, and that a compact non-zero warm start outperforms both zero and wide random initializations."
    ),
    wr.PanelGrid(runsets=init_ablation_runs, panels=init_ablation_plots, hide_run_sets=True),
]

## F3 — First Canon Layer Alone Recovers Most of the Benefit; Canon-AC ≈ Canon-ABCD

In [159]:
layer_ablation_run_names = [
    "text_llama_bs_128_seq_len_2048_0.1.206",                                                    # Baseline
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.207",               # Canon ABCD — all layers
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.195",               # Canon ABCD — first layer only
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.197",               # Canon ABCD — last layer only
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.138",               # Canon AC — all layers
]
layer_ablation_runs = [runsets_factory.filter_by_run_names(layer_ablation_run_names)]

layer_ablation_plots = [loss_lineplot(title="Canon Layer Placement Ablation")]

report.blocks += [
    wr.H2("F3 — First Canon Layer Alone Recovers Most of the Benefit; Canon-AC ≈ Canon-ABCD"),
    wr.P(
        "Last-layer Canon is slightly worse than baseline. "
        "First-layer Canon alone nearly matches full Canon-ABCD. "
        "Canon-AC (only the pre-attention and pre-MLP insertion points) "
        "matches Canon-ABCD throughout training, confirming that Canon-B and Canon-D "
        "(inside Q/K/V projection and inner-MLP) contribute negligibly. "
    ),
    wr.PanelGrid(runsets=layer_ablation_runs, panels=layer_ablation_plots, hide_run_sets=True),
]

## F4 — Normalization Type Interacts with Canon

In [160]:
run_names_peri = [
    "text_llama_bs_128_seq_len_2048_0.1.212",                                       # Peri-LN Llama
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.213",   # Peri-LN Canon
]
run_names_pre = [
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.207",   # Pre-LN Canon
    "text_llama_bs_128_seq_len_2048_0.1.206",                                       # Pre-LN Llama
]
run_names_post = [
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.196",   # Post-LN Canon
    "text_llama_bs_128_seq_len_2048_0.1.198",                                       # Post-LN Llama
]
norm_all_run_names = run_names_peri + run_names_post + run_names_pre

norm_runs = [runsets_factory.filter_by_run_names(norm_all_run_names)]
norm_runs_pre = [runsets_factory.filter_by_run_names(run_names_pre)]
norm_runs_post = [runsets_factory.filter_by_run_names(run_names_post)]
norm_runs_peri = [runsets_factory.filter_by_run_names(run_names_peri)]

norm_loss_plots = [loss_lineplot(title="Norm Type Ablation — all configs")]

# Per-norm-type gradient plots
grad_norm_plots = [grad_contrib(i) for i in range(8)]
grad_norm_plots = realign_grid(grad_norm_plots, num_columns=4, total_w=25, plot_h=7)


report.blocks += [
    wr.H2("F4 — Canon improves performance with any normailzation location"),
    wr.P(
        "Three normalization positions alongside Canon: "
        "Pre-LN (norm before attention/MLP — standard in Llama), "
        "Post-LN (norm after residual), and "
        "Peri-LN (norm around both input and output of sublayer)."
    ),
    wr.P(
        "Canon improves the convergence speed along any normalization placement with biggest benefit on pre-norm"
    ),
    wr.PanelGrid(runsets=norm_runs, panels=norm_loss_plots, hide_run_sets=True),
    wr.H3("Gradient contributions by layer and norm type"),
    wr.P("Pre-LN:"),
    wr.PanelGrid(runsets=norm_runs_pre, panels=grad_norm_plots, hide_run_sets=True),
    wr.P("Post-LN:"),
    wr.PanelGrid(runsets=norm_runs_post, panels=grad_norm_plots, hide_run_sets=True),
    wr.P("Peri-LN:"),
    wr.PanelGrid(runsets=norm_runs_peri, panels=grad_norm_plots, hide_run_sets=True),
]

## F5 — Larger Canon Kernel Size Improves Text Loss

In [161]:
ks_ablation_run_names = [
    "canon_text_0.1.130",                                                             # Baseline (no Canon)
    "canon_text_0.1.117",                                                             # Canon ks=1
    "text_ks_2_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.164",  # Canon ks=2
    "text_ks_4_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.147",  # Canon ks=4
    "text_ks_6_const_var_sqr_init_with_residual_trainable_bs_128_seq_len_2048_0.1.148",  # Canon ks=6
]
ks_ablation_runs = [runsets_factory.filter_by_run_names(ks_ablation_run_names)]

ks_ablation_plots = [loss_lineplot(title="Canon Kernel Size Ablation")]

report.blocks += [
    wr.H2("F5 — Kernel Size ablation"),
    wr.P(
        "The conv1d kernel size controls how many past tokens Canon can mix. "
        "Four kernel sizes were compared against the baseline using const_var_sqr initialization."
    ),
    wr.P(
        "loss ranking: "
        "ks=6 < ks=4 < ks=2 < baseline < ks=1. "
        "A kernel of 1 is worse than no Canon at all — it adds parameters without mixing "
        "any neighbouring tokens, introducing noise without benefit. "
        "Larger kernels consistently improve loss, with ks=6 being best, "
    ),
    wr.PanelGrid(runsets=ks_ablation_runs, panels=ks_ablation_plots, hide_run_sets=True),
]

## F6 — Canon Has Two Roles: Information Mixing and Implicit Normalisation

In [162]:
dynamics_run_names = [
    "text_ks_4_default_init_with_residual_trainable_bs_128_seq_len_2048_0.1.186",   # Canon all layers
    "text_llama_bs_128_seq_len_2048_0.1.189",                                       # Baseline
]
dynamics_runs = [runsets_factory.filter_by_run_names(dynamics_run_names)]

# Canon weights layerwise (A, B, C, D × layers 0, 3, 7)
canon_weights_plots = [
    canon_weight_fixed_layer("A", 0), canon_weight_fixed_layer("A", 3), canon_weight_fixed_layer("A", 7),
    canon_weight_fixed_layer("B", 0), canon_weight_fixed_layer("B", 3), canon_weight_fixed_layer("B", 7),
    canon_weight_fixed_layer("C", 0), canon_weight_fixed_layer("C", 3), canon_weight_fixed_layer("C", 7),
    canon_weight_fixed_layer("D", 0), canon_weight_fixed_layer("D", 3), canon_weight_fixed_layer("D", 7),
]

# RMS ratio and residual RMS
rms_plots = [
    canon_rms_ratio("A"), canon_rms_ratio("B"),
    canon_rms_ratio("C"), canon_rms_ratio("D"),
    residual_rms("ffn"), residual_rms("attn"),
]
rms_plots = realign_grid(rms_plots, num_columns=2, total_w=25, plot_h=7)

# Gradient contributions
grad_plots = [grad_contrib(i) for i in range(8)]
grad_plots = realign_grid(grad_plots, num_columns=4, total_w=25, plot_h=7)

# Outlier features (kurtosis)
outlier_plots = [outlier_features_kurtosis(i) for i in range(8)]
outlier_plots = realign_grid(outlier_plots, num_columns=4, total_w=25, plot_h=7)

# Cosine similarity Canon output vs input
cos_plots = [cos_sim(ct) for ct in ["A", "B", "C", "D"]]
cos_plots = realign_grid(cos_plots, num_columns=2, total_w=25, plot_h=7)

report.blocks += [
    wr.H2("F6 — Canon Has Two Roles: Information Mixing and Implicit Normalisation"),
    wr.H3("1. Information mixing — higher weights on farther tokens"),
    wr.P(
        "Canon weight magnitudes consistently increase with shift distance "
        "(shift 3 > shift 2 > shift 1 > shift 0 for most layers)"
    ),
    wr.PanelGrid(runsets=dynamics_runs, panels=canon_weights_plots, hide_run_sets=True),

    wr.H3("2. Representation upscaling — Canon amplifies deeper layers more"),
    wr.P(
        "The RMS ratio (Canon output / Canon input) shows that representations are "
        "upscaled, and deeper layers (closer to the LM head) are scaled more than "
        "early layers."
    ),
    wr.PanelGrid(runsets=dynamics_runs, panels=rms_plots, hide_run_sets=True),

    wr.H3("3. Gradient redistribution — Canon smooths gradient flow"),
    wr.P(
        "Without Canon, gradient contributions increase monotonically from layer 0 to layer 7 "
        "(e.g. ~6 at layer 0, ~20 at layer 7). With Canon, gradients are much more uniform "
        "across layers (~8–12). "
    ),
    wr.PanelGrid(runsets=dynamics_runs, panels=grad_plots, hide_run_sets=True),

    wr.H3("4. Cosine similarity — more mixing in early layers and Canon-A, C"),
    wr.P(
        "Cosine similarity between Canon output and input is lowest (most mixing) "
        "in early layers and in Canon-A and Canon-C positions. "
        "Deeper layers show higher similarity (less mixing), consistent with "
        "the weight magnitude analysis. Canon-B and Canon-D contribute less "
        "directional change, suggesting the Q/K/V and inner-MLP paths are less "
        "critical for the mixing effect."
    ),
    wr.PanelGrid(runsets=dynamics_runs, panels=cos_plots, hide_run_sets=True),
]

In [163]:
report.save()
print("Report saved.")

wandb: Saved report to: https://wandb.ai/kirill456z/physics4llm/reports/Project-Findings---Midterm-report--VmlldzoxNjM3NTU5NA==


Report saved.
